In [ ]:
import pymysql
import pandas as pd

# 数据库连接
conn = pymysql.connect(
    host='###',
    port=3306,
    user='###',
    password='###',
    database='ch00',
    charset='utf8mb4'
)

# 设定目标日计算天数
target_date = '2025-05-21'
n_days=40

# 步骤1：获取目标日中证500成分股的 ticker 列表
sql_get_tickers = f"""
SELECT ticker
FROM idx_wgt
WHERE idx = '000905' AND dt = '{target_date}'
"""
tickers_df = pd.read_sql(sql_get_tickers, conn)
tickers = tickers_df['ticker'].tolist()
tickers_str = ','.join([f"'{t}'" for t in tickers])  # 用于SQL IN语句

# 步骤2：获取这些股票过去40天的行情数据
sql_price_data = f"""
SELECT dt, ticker, last AS close_price
FROM adjprc
WHERE ticker IN ({tickers_str})
  AND dt <= '{target_date}'
ORDER BY dt DESC
LIMIT {n_days * len(tickers)}  -- 估算最大行数,Python中再处理实际交易日
"""
price_df = pd.read_sql(sql_price_data, conn)

# 步骤3：获取目标日的权重
sql_weight_data = f"""
SELECT ticker, wgt
FROM idx_wgt
WHERE idx = '000905' AND dt = '{target_date}'
"""
wgt_df = pd.read_sql(sql_weight_data, conn).set_index('ticker')

# 步骤4：清洗 + 选取最近40个交易日
price_df['dt'] = pd.to_datetime(price_df['dt'])
latest_dates = price_df['dt'].drop_duplicates().sort_values(ascending=False).head(n_days)
price_df = price_df[price_df['dt'].isin(latest_dates)]

# 重新整理为价格矩阵
price_matrix = price_df.pivot(index='dt', columns='ticker', values='close_price').sort_index()

# 关闭数据库
conn.close()

# 输出检查
print("价格矩阵：", price_matrix.shape)
print("权重向量：", wgt_df.shape)
print(price_matrix)
print(wgt_df)



In [ ]:
import numpy as np

# Step 1: 计算对数收益率
log_returns = np.log(price_matrix / price_matrix.shift(1)).dropna()

# Step 2: 计算协方差矩阵（用于最小方差优化）
cov_matrix = log_returns.cov()

# Step 3: 计算平均对数收益率（用于目标收益优化）
mean_returns = log_returns.mean()

log_tickers = log_returns.columns
w_index = wgt_df.loc[log_tickers].values.flatten()

print(w_index.shape)
# 输出检查
print("协方差矩阵维度:", cov_matrix.shape)
print("平均收益率维度:", mean_returns.shape)
print(mean_returns)


In [ ]:
import cvxpy as cp

##限定方差前提下计算最大收益率##

# 数据准备
n = len(mean_returns)
mu = mean_returns.values          # 收益均值向量
Sigma = cov_matrix.values         # 协方差矩阵
risk_limit = 0.00004            # 设定最大方差

# 定义优化变量
x = cp.Variable(n)

# 定义目标函数：最大化收益
objective = cp.Maximize(mu @ x)

# 约束条件
constraints = [
    cp.quad_form(x, Sigma) <= risk_limit,  # 方差约束
    cp.sum(x) == 1,                         # 权重和为1
    x >= 0                                  # 不允许空头
]

# 定义并求解问题
prob = cp.Problem(objective, constraints)
prob.solve(solver=cp.MOSEK)

# 输出最优解
opt_weight = x.value

print("最优收益组合权重（前10项）：")
for i, ticker in enumerate(cov_matrix.columns[0:10]):
    print(f"{ticker}: {opt_weight[i]:.4f}")

# （可选）转换为 DataFrame
opt_df = pd.DataFrame({
    'ticker': cov_matrix.columns,
    'optimized_weight': opt_weight
}).set_index('ticker')

top10 = opt_df.sort_values(by='optimized_weight', ascending=False).head(10)

print("最优组合中权重最大的前10只股票：")
print(top10)

In [ ]:
# 合并两个 DataFrame
combined_df = opt_df.join(wgt_df, how='inner')  # inner 确保只保留两边都存在的 ticker

print(combined_df.shape)
# 重命名列以便清晰区分
combined_df = combined_df.rename(columns={'wgt': 'index_weight'})

# 按优化权重从高到低排序
combined_df = combined_df.sort_values(by='optimized_weight', ascending=False)

# 查看前几行
print(combined_df.head(20))
